# 05 — Baseline de modelado

Modelo baseline para predecir `rhythm_label` a partir de las features generadas en `04_windowing_and_feature_engineering.ipynb`.

**Reglas obligatorias**

- Split train/test **por `case_id`** (`GroupKFold` / `GroupShuffleSplit`).
- `beat_type` **no** entra como feature.
- Métricas macro y reporte por clase como salida principal.

**Alcance**

Este notebook entrena un baseline pequeño únicamente para validar el flujo. **No** se reportan cifras finales hasta ejecutar las celdas con datos reales.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src.modeling import (
    assert_no_forbidden_features,
    build_logreg_pipeline,
    build_rf_pipeline,
    make_group_kfold,
    make_group_split,
)
from src.evaluation import compute_macro_metrics, per_class_report, confusion_matrix_df
from src.utils import get_logger, set_seed

set_seed(config.RANDOM_SEED)
logger = get_logger("nb05")
sns.set_theme(context="notebook", style="whitegrid")

## 2. Carga de la tabla de features

In [ ]:
features_path = config.PROCESSED_DIR / "features_baseline.parquet"
if not features_path.exists():
    raise FileNotFoundError(
        f"No existe {features_path}. Ejecuta primero 04_windowing_and_feature_engineering.ipynb."
    )

df = pd.read_parquet(features_path)
df = df.dropna(subset=[config.TARGET_COLUMN])
print("Shape:", df.shape)
df.head()

## 3. Preparación de X, y y grupos

Se excluyen explícitamente columnas no-predictoras: `case_id`, `rhythm_label`, `beat_type`, `bad_signal_quality`, además de identificadores de ventana.

In [ ]:
non_feature_cols = set(config.FORBIDDEN_FEATURE_COLUMNS) | {"beat_index", "start_sample", "end_sample"}
feature_cols = [c for c in df.columns if c not in non_feature_cols]

# Validación: si alguna columna prohibida se cuela como feature se aborta.
assert_no_forbidden_features(feature_cols)

X = df[feature_cols].to_numpy()
y = df[config.TARGET_COLUMN].to_numpy()
groups = df[config.CASE_ID_COLUMN].to_numpy()

print("X shape:", X.shape)
print("y único:", np.unique(y, return_counts=True))
print("Grupos únicos:", np.unique(groups).shape[0])

## 4. Split simple por `case_id`

In [ ]:
train_idx, test_idx = make_group_split(X, y, groups, test_size=0.2, random_state=config.RANDOM_SEED)
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# Verificación dura: ningún `case_id` puede aparecer en ambos splits.
assert set(groups[train_idx]).isdisjoint(set(groups[test_idx])), "Fuga de grupo entre train y test."

print("Train:", X_train.shape, "Test:", X_test.shape)

## 5. Entrenamiento de pipelines baseline

In [ ]:
pipelines = {
    "logreg": build_logreg_pipeline(),
    "random_forest": build_rf_pipeline(),
}

results = {}
for name, pipe in pipelines.items():
    logger.info("Entrenando %s...", name)
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    results[name] = {
        "model": pipe,
        "y_pred": y_pred,
        "metrics": compute_macro_metrics(y_test, y_pred),
    }
    print(name, results[name]["metrics"])

## 6. Reporte por clase y matriz de confusión

In [ ]:
for name, info in results.items():
    print(f"=== {name} ===")
    print(per_class_report(y_test, info["y_pred"]))
    cm = confusion_matrix_df(y_test, info["y_pred"], normalize="true")
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", ax=ax)
    ax.set_title(f"Matriz de confusión normalizada ({name})")
    ax.set_xlabel("Predicho")
    ax.set_ylabel("Real")
    plt.tight_layout()
    plt.show()

## 7. Validación cruzada por grupos (`GroupKFold`)

Reporte fold-a-fold para mostrar variabilidad entre cortes por `case_id`.

In [ ]:
fold_metrics = []
pipe = build_logreg_pipeline()
for i, (tr, te) in enumerate(make_group_kfold(X, y, groups, n_splits=config.DEFAULT_N_SPLITS)):
    pipe.fit(X[tr], y[tr])
    y_pred = pipe.predict(X[te])
    m = compute_macro_metrics(y[te], y_pred)
    m["fold"] = i
    fold_metrics.append(m)

cv_df = pd.DataFrame(fold_metrics).set_index("fold")
cv_df

## 8. Próximos pasos

- Iterar sobre features adicionales (frecuenciales, morfológicas).
- Probar estrategias de balanceo (`class_weight`, sobremuestreo).
- Comparar contra `XGBoost` (`src.modeling.build_xgb_pipeline`).
- Documentar limitaciones encontradas en `reports/`.